# analytic filters

this notebook builds cheap physics-based filters that screen rocket designs before they ever reach the KSP simulator.

## context

the structural validator (goal 0) tells us whether a rocket is *possible* — valid parts, connected tree, compatible propellants. it says nothing about whether the rocket can actually fly.

KSP is slow. we can't afford to simulate every candidate design. the analytic filters sit between the validator and KSP, using back-of-envelope physics to reject designs that are obviously incapable of reaching the mission target. only designs that pass all filters get handed to KSP.

## the three filters

**1. thrust-to-weight ratio (TWR)**

$$TWR = \frac{F_{thrust}}{m_{total} \cdot g}$$

if TWR < 1.0 at launch, the rocket cannot lift off. this is a hard physical impossibility — no simulation needed. we'll use a slightly higher threshold (~1.2) to avoid designs that technically lift off but perform poorly.

**2. delta-v estimate**

$$\Delta v = \sum_{\text{stages}} I_{sp} \cdot g_0 \cdot \ln\left(\frac{m_{wet}}{m_{dry}}\right)$$

using the tsiolkovsky rocket equation, applied per stage and summed. if total delta-v is below the mission budget (3400 m/s for low Kerbin orbit), the rocket cannot reach the target regardless of how it's flown.

**3. burn time**

$$t_{burn} = \frac{m_{propellant}}{\dot{m}} = \frac{m_{propellant} \cdot I_{sp} \cdot g_0}{F_{thrust}}$$

a sanity check: does each stage burn for a meaningful duration? a stage that exhausts in 2 seconds is probably a design artifact, not a useful stage.

## what we need from the parts library

all three filters are computable from data we already have:

| quantity | source |
|----------|--------|
| engine thrust (kN) | `part['engine']['max_thrust']` |
| engine Isp (s) | `part['engine']['isp']` |
| part dry mass (t) | `part['mass']` |
| tank resource amounts | `part['resources']` |
| resource mass (t/unit) | scraped from `ResourcesGeneric.cfg` |

## plan

1. scrape KSP resource densities from `ResourcesGeneric.cfg`
2. write `compute_twr(rocket_dict, parts_by_name, resource_lookup)` — total thrust vs total mass
3. write `compute_delta_v(rocket_dict, parts_by_name, resource_lookup)` — per-stage rocket equation, summed
4. write `compute_burn_time(rocket_dict, parts_by_name, resource_lookup)` — per-stage propellant mass / mass flow rate
5. write `filter_rocket(rocket_dict, parts_by_name, resource_lookup, goal)` — runs all three, returns pass/fail with reasons
6. test on the toy rocket and some obviously bad designs
7. extract to `src/filters.py`

parse resources config and make dict of resources, keyed by name and with a value of its density

In [1]:
from pathlib import Path
from src.scraper import parse_cfg
import json
import numpy as np

### parts library
parts_json_file = Path('../data/parts_library.json')
with open(parts_json_file) as f:
    parts_library = json.load(f)

# parts dictionary by name
parts_by_name = {p['name']: p for  p in parts_library }

# toy example (could have loaded but lazy)
rocket_example = {
    "parts": [
        {"id": "pod_0",  "type": "mk1-3pod", "parent": None},
        {"id": "tank_0", "type": "fuelTank", "parent": "pod_0", "attach_node": "bottom"  },
        {"id": "eng_0",  "type": "liquidEngine", "parent": "tank_0", "attach_node": "bottom"},
    ],
    "stages": {"eng_0": 0}
}

# parts list by name
parts = [part["type"] for part in rocket_example["parts"]]

get kerbal part physics resources (fuel density)

In [2]:
resources_file = Path('/Users/moss/Library/Application Support/Steam/steamapps/common/Kerbal Space Program/GameData/Squad/Resources/ResourcesGeneric.cfg')
with open(resources_file, encoding='utf-8-sig') as f:
    raw = f.read()
resources_data = parse_cfg(raw)

resource_lookup = {}
for child in resources_data['_children']:
    resource_lookup[child['name']] = {'density':float(child['density'])}

print(resource_lookup)


{'LiquidFuel': {'density': 0.005}, 'Oxidizer': {'density': 0.005}, 'SolidFuel': {'density': 0.0075}, 'MonoPropellant': {'density': 0.004}, 'XenonGas': {'density': 0.0001}, 'ElectricCharge': {'density': 0.0}, 'IntakeAir': {'density': 0.005}, 'EVA Propellant': {'density': 0.005}, 'Ore': {'density': 0.01}, 'Ablator': {'density': 0.001}}


calculate total mass

In [3]:
# print(parts_by_name['liquidEngine'].keys())
# print(parts_by_name['liquidEngine']['mass_t'])
# print(parts_by_name['fuelTank'].keys())
# print(parts_by_name['fuelTank']['mass_t'])
# print(type(parts_by_name['fuelTank']['mass_t']))
# print(parts_by_name['liquidEngine']['resources'])
# print(parts_by_name['fuelTank']['resources'])

def get_total_mass(parts_list: list,
                   parts_dict: dict,
                   fuel_lookup: dict):

    total_dry_mass = 0
    for part in parts_list:
        mass = parts_dict[part]['mass_t']
        total_dry_mass += mass

    total_fuel_mass = 0
    for part in parts_list:
        part_entry = parts_dict[part]
        if part_entry['resources'] is not None:
            for resource in part_entry['resources'].keys():
                units = part_entry['resources'][resource]
                conversion = fuel_lookup[resource]['density']
                mass = units * conversion
                total_fuel_mass += mass

    total_mass = total_dry_mass + total_fuel_mass
    return total_mass

get_total_mass(parts, parts_by_name, resource_lookup)

6.22

calculate thrust

In [4]:
# parts_by_name['liquidEngine']['engine']
def calculate_thrust(parts_list: list,
                   parts_dict: dict):

    total_thrust = 0
    for part in parts_list:
        part_entry = parts_dict[part]
        if part_entry['engine'] is not None:
            thrust = part_entry['engine']['max_thrust_kn']
            total_thrust += thrust

    return total_thrust

calculate_thrust(parts, parts_by_name)

240.0

In [5]:
def calculate_twr(parts_list: list,
                  parts_dict: dict,
                  fuel_lookup: dict,
                  g_const: float = 9.80665):

    mass = get_total_mass(parts_list, parts_dict, fuel_lookup)
    thrust = calculate_thrust(parts_list, parts_dict)
    twr = thrust / (mass * g_const)

    return twr

calculate_twr(parts, parts_by_name, resource_lookup)

3.934596320172071

In [6]:
# two-stage rocket
# stage 1 fires first (booster): eng_booster burns tank_booster, then decoupler fires and drops booster
# stage 0 fires last (upper):    eng_upper burns tank_upper

staged_rocket = {
    "parts": [
        {"id": "pod_0",        "type": "mk1-3pod",    "parent": None},
        {"id": "tank_upper",   "type": "fuelTank",    "parent": "pod_0",      "attach_node": "bottom"},
        {"id": "eng_upper",    "type": "liquidEngine", "parent": "tank_upper", "attach_node": "bottom"},
        {"id": "decoupler_0",  "type": "Decoupler_1", "parent": "eng_upper",  "attach_node": "bottom"},
        {"id": "tank_booster", "type": "fuelTank",    "parent": "decoupler_0","attach_node": "bottom"},
        {"id": "eng_booster",  "type": "liquidEngine", "parent": "tank_booster","attach_node": "bottom"},
    ],
    "stages": {
        "eng_booster":  1,   # fires first
        "decoupler_0":  1,   # jettisons with booster stage
        "eng_upper":    0,   # fires last
    }
}

print(staged_rocket)

{'parts': [{'id': 'pod_0', 'type': 'mk1-3pod', 'parent': None}, {'id': 'tank_upper', 'type': 'fuelTank', 'parent': 'pod_0', 'attach_node': 'bottom'}, {'id': 'eng_upper', 'type': 'liquidEngine', 'parent': 'tank_upper', 'attach_node': 'bottom'}, {'id': 'decoupler_0', 'type': 'Decoupler_1', 'parent': 'eng_upper', 'attach_node': 'bottom'}, {'id': 'tank_booster', 'type': 'fuelTank', 'parent': 'decoupler_0', 'attach_node': 'bottom'}, {'id': 'eng_booster', 'type': 'liquidEngine', 'parent': 'tank_booster', 'attach_node': 'bottom'}], 'stages': {'eng_booster': 1, 'decoupler_0': 1, 'eng_upper': 0}}


⏺ Clean and correct. TWR of ~3.9 for this toy rocket checks out — 240 kN of thrust against 6.22 tonnes is a very powerful little rocket, which makes sense (the LV-T45 is a strong engine relative to a minimal 3-part stack).

  The units work out too: kN ÷ (tonnes × m/s²) → kN ÷ kN → dimensionless. No conversion needed.

  The g_const default parameter is a nice touch. Everything looks good — on to compute_delta_v. This one is more interesting because it has to be done per stage.

  Before you write it, think about what "per stage" means in terms of the data we have. The stages dict maps part ids to stage numbers — but only for engines and decouplers. For the delta-v calculation, you need to know for each stage: which
  engines are firing, and what mass of propellant do they consume.

  The key question: how do you figure out which fuel tanks "belong to" a given stage? In KSP, a stage's engines consume fuel from all tanks above them in the stack. Given our part tree structure, how would you identify which tanks are above a
  given engine?

  Have a think on that before writing anything — it's the crux of the per-stage calculation.



In [11]:
def compute_delta_v(rocket_dict: dict,
                    parts_list: list,
                    parts_dict: dict,
                    resource_lookup: dict):

    # parts lookup by id
    id_to_type = {p['id']: p['type'] for p in rocket_dict['parts']}
    #parent lookup by id for trace
    id_to_parent = {p['id']: p['parent'] for p in rocket_dict['parts']}

    for part_id, stage_num in rocket_dict['stages'].items():
        part_type = id_to_type[part_id]
        if parts_dict[part_type]['engine'] is not None:
            propellants = parts_dict[part_type]['engine']['propellants'].keys()
            current = part_id

            fuel_mass = 0
            while current is not None:
                current_type = id_to_type[current]
                if current_type.startswith('Decoupler'):
                    break
                part_resources = parts_dict[current_type]['resources']
                if part_resources is not None:
                    for resource, amount in part_resources.items():
                        if resource in propellants:
                            fuel_mass += amount * resource_lookup[resource]['density']
                            print(f'current fuel mass is {fuel_mass}')
                current = id_to_parent[current]






compute_delta_v(staged_rocket, parts, parts_by_name, resource_lookup)


current fuel mass is 0.9
current fuel mass is 2.0
current fuel mass is 0.9
current fuel mass is 2.0
